In [ ]:
import pandas as pd
import numpy as np

# =========================
# CONFIG
# =========================
# Replace PXX with the verified public participant code only after the
# private legacy-name -> public-code crosswalk has been checked.
PARTICIPANT_ID = "PXX"

INPUT_PATH = f"{PARTICIPANT_ID}.csv"
CLEANED_PATH = f"moving_avg_fixed_repeats_{PARTICIPANT_ID}.csv"

WINDOW_SIZE = 5
CENTERED = True
META_CANDIDATES = {"elements", "element", "id", "time", "timestamp"}

# =========================
# Robust CSV loader
# =========================
def read_csv_robust(path: str) -> pd.DataFrame:
    for sep in [";", ",", "\t"]:
        try:
            df_try = pd.read_csv(path, sep=sep)
            if df_try.shape[1] > 1:
                print(f"Loaded with sep='{sep}' -> shape={df_try.shape}")
                return df_try
        except Exception:
            continue
    df = pd.read_csv(path)
    print(f"Loaded with default separator -> shape={df.shape}")
    return df

# =========================
# Rolling mean filler
# =========================
def fill_na_with_rolling_mean(s: pd.Series, w: int, centered: bool) -> pd.Series:
    roll = s.rolling(window=w, min_periods=1, center=centered).mean()
    return s.fillna(roll)

# =========================
# MAIN
# =========================
df_raw = read_csv_robust(INPUT_PATH)

meta_cols = [c for c in df_raw.columns if str(c).strip().lower() in META_CANDIDATES]
df_meta = df_raw[meta_cols].copy() if meta_cols else None

df = df_raw.drop(columns=meta_cols, errors="ignore").copy()
df = df.apply(pd.to_numeric, errors="coerce")

# Replace consecutive repeated values with NaN.
for col in df.columns:
    dup_mask = df[col].eq(df[col].shift(1))
    df.loc[dup_mask, col] = np.nan

# Fill NaNs with a moving average.
df = df.apply(lambda s: fill_na_with_rolling_mean(s, WINDOW_SIZE, CENTERED))

# Safety pass for edge/long gaps.
df = df.interpolate(method="linear", axis=0, limit_direction="both")
df = df.replace([np.inf, -np.inf], np.nan)
df = df.ffill().bfill()

df_out = pd.concat([df_meta, df], axis=1) if df_meta is not None else df
df_out.to_csv(CLEANED_PATH, index=False)

print("Saved:", CLEANED_PATH)
print("Remaining NaN count:", int(df_out.isna().sum().sum()))
print(df_out.head())


import pandas as pd
import numpy as np

# =========================
# LOAD
# =========================
df = pd.read_csv(CLEANED_PATH)
EPS = 1e-12

bands = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
band_cols = [c for c in df.columns if any(c.startswith(b) for b in bands)]
df[band_cols] = df[band_cols].apply(pd.to_numeric, errors="coerce")

baseline_df = df.loc[
    df["Elements"].astype(str).str.lower().eq("baseline"), band_cols
].copy()
cond_df = df.loc[
    df["Elements"].astype(str).str.lower().isin(["empty", "withpeople"])
].copy()

if baseline_df.empty:
    raise ValueError("No baseline rows found. Check 'Elements' values.")
if cond_df.empty:
    raise ValueError("No condition rows found for Empty/WithPeople.")

baseline_mean = baseline_df.mean(axis=0, skipna=True)

cond_df_band = cond_df[band_cols].copy()
cond_df_blpct = (cond_df_band - baseline_mean) / (baseline_mean.abs() + EPS)

for c in band_cols:
    cond_df[f"{c}_BLpct"] = cond_df_blpct[c]

required = [
    "Alpha_AF7_BLpct", "Alpha_AF8_BLpct",
    "Beta_AF7_BLpct", "Beta_AF8_BLpct",
]
missing = [c for c in required if c not in cond_df.columns]
if missing:
    raise KeyError(f"Missing required columns for A/V: {missing}")

Alpha_L = cond_df["Alpha_AF7_BLpct"]
Alpha_R = cond_df["Alpha_AF8_BLpct"]
Beta_L = cond_df["Beta_AF7_BLpct"]
Beta_R = cond_df["Beta_AF8_BLpct"]

# Baseline-referenced descriptive indices.
cond_df["Valence_BL"] = Alpha_L - Alpha_R
cond_df["Arousal_BL"] = (Beta_L + Beta_R) / (Alpha_L + Alpha_R + EPS)

AV_PATH = f"arousal_valence_baseline_referenced_{PARTICIPANT_ID}.csv"
cond_df.to_csv(AV_PATH, index=False)

print("Saved:", AV_PATH)
print(cond_df[["Elements", "Arousal_BL", "Valence_BL"]].head())


In [ ]:
import pandas as pd
from scipy import stats

# =========================
# LOAD
# =========================
df = pd.read_csv(AV_PATH)

# =========================
# FILTER: WITHPEOPLE CONDITION
# =========================
people_df = df.loc[df["Elements"] == "WithPeople"].copy()

if people_df.empty:
    raise ValueError("No 'WithPeople' condition rows found.")

desc_df = (
    people_df[["Arousal_BL", "Valence_BL"]]
    .agg(["mean", "std", "median", "count"])
    .transpose()
    .reset_index()
    .rename(columns={"index": "Metric"})
)

desc_output = f"{PARTICIPANT_ID}_people_descriptive_vs_baseline.csv"
desc_df.to_csv(desc_output, index=False)

ttest_results = []
for metric in ["Arousal_BL", "Valence_BL"]:
    values = people_df[metric].dropna()
    if len(values) < 2:
        continue
    t, p = stats.ttest_1samp(values, 0.0)
    ttest_results.append({
        "Condition": "WithPeople",
        "Metric": metric,
        "Mean": values.mean(),
        "Std": values.std(),
        "t_stat": t,
        "p_value": p,
        "Direction_vs_Baseline": "Increase" if values.mean() > 0 else "Decrease",
    })

ttest_df = pd.DataFrame(ttest_results)
ttest_output = f"{PARTICIPANT_ID}_people_ttest_vs_baseline.csv"
ttest_df.to_csv(ttest_output, index=False)

print("Saved:", desc_output)
print("Saved:", ttest_output)
